# BUILDING A WEB_SUMMARIZER AND TRANSLATOR.

- Create a webscrapper function.
- call openai Api to summarize the site in different languages


# ACCESSING MY API KEY


In [ ]:
import os, dotenv
from openai import OpenAI
dotenv.load_dotenv(override = True)
my_key = os.getenv("MY_OPENAI_API_KEY")
client = OpenAI(api_key = my_key)

## MY SCRAPPER FUNCTION


In [ ]:
import requests
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

def summarize_site(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }  #to bypass security an dprevent an empty response, we pose as a human being
    response = requests.get(url, headers=headers)
    webcode = response.text
    soup = BeautifulSoup(webcode, "html.parser")
    # 1. Create a blank "Summary" string to hold our structured text
    summary = ""

    # 2. Add the Title to the summary
    title_box = soup.find("h1")                          #check for a title tag "h1"
    if title_box:                                        #if u find a title tag                                                      
        summary += f"# {title_box.text}\n\n"             #do not print yet, add it to our container called "summary" (We add "#" for Markdown heading style and "\n\n" for space)
    else:
        summary += "# (No Title Found)\n\n"

    # 3. Add the Paragraphs to the Summary
    p_tags = soup.find_all("p")                          # find the paragraph tags "p". note; it puts it in a list at default.
    for p in p_tags:                                     # we have to take the orange one after the other from the basket to be able to peel the orange
        summary += f"{p.text}\n\n"                       # give me each paragraph but remove the tags give the texts ie, ".text"
    
    display(Markdown(summary))
    return summary                    # THE FINALE: Display the 'summary' we built, NOT the 'soup.text'
site_info = summarize_site("https://en.wikipedia.org/wiki/Pied_butcherbird")

## CALLING OPENAI TO TRANSLATE


In [ ]:
from openai import OpenAI
client = OpenAI()
my_system_prompt = "you are a polyglot assistant that summarizes and translates websites in any language requested"
my_user_prompt = "you will summarize and translate the contents of the webpage provided into Nigerian pidgin" 

def translate_site(url):
    # 1. Fetch the content using your scraper function
    

    # 2. Combine your instructions with the actual text
    # We use an f-string to put the site_info inside the user prompt
    full_user_prompt = f"{my_user_prompt} \n\n Here is the content: {summarize_site(url)}"

    my_messages = [
        {"role": "system", "content": my_system_prompt},
        {"role": "user", "content": full_user_prompt} # Now the AI has the data!
    ]
    
    # 3. Call OpenAI (Use a valid model like 'gpt-4o' or 'gpt-4o-mini')
    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=my_messages
    )
    
    return response.choices[0].message.content

result = translate_site("https://edwarddonner.com")
print(result)

# MY SECOND ATTEMPT

## BUILDING THE SCRAPPER USING BEAUTIFULSOUP


In [ ]:
import requests
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

def summarize_web(url):
  headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
  
  answer = requests.get(url, headers=headers)
  web = answer.text
  soup = BeautifulSoup(web, "html.parser")
  summary = ""
  title = soup.find("h1")
  if title:
    summary += f"# TITLE: {title.text}\n\n"
  else:
    print("no Title found on page")

  p_tags = soup.find_all("p")
  for p in p_tags:
    summary += f"{p.text}\n\n"
  
  return summary


## CALLING THE OPENAI API


In [ ]:
from openai import OpenAI
client = OpenAI()

# 1. Define the URL here so the computer knows what it is!
target_url = "https://en.wikipedia.org/wiki/Pied_butcherbird"

# 2. Call the function and save the result to a variable
translated = summarize_web(target_url)

my_system_prompt = "you are a smart virtual assistant"
my_user_prompt = f"you are going to translate the content of this web page into pidgin english. leave no information out. here is the content{translated}!"

my_messages = [
  {"role" : "system" , "content" : my_system_prompt} ,
  {"role" : "user" , "content" : my_user_prompt}
]

response = client.chat.completions.create(model = "gpt-4o", messages = my_messages)
result = response.choices[0].message.content

display(Markdown(result))


# BUILDING A SCRAPPER WITH PLAYWRIGHT

- although it doesnt work with ipynb. it is prefered to write ur code in a py file.


In [ ]:
#BUILD THE SCRAPPER
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

def summarize_dynamic_site(url):  # no more 'async'
    with sync_playwright() as p:                                          
        browser = p.chromium.launch(headless=True)      # launch the robot
        page = browser.new_page()                        # the robot opens a new page
        page.goto(url)                                   # the robot goes to the site
        page.wait_for_load_state("networkidle")          # wait till page loads completely
        full_html = page.content()                       # grab the page contents
        
        browser.close()                                  # shut down to preserve RAM

    soup = BeautifulSoup(full_html, "html.parser") 
    summary = ""
    title = soup.find("h1")
    if title:
        summary += f"TITLE: {title.text}\n\n"
    else:
        summary += "this text has got no heading\n\n"    
    p_tags = soup.find_all("p")
    for p in p_tags:
        summary += f"{p.text}\n\n"
    return summary

In [ ]:
#CALL THE API
from openai import OpenAI
client = OpenAI()

url = "https://khimoai.my.canva.site"
to_be_translated = summarize_dynamic_site(url)  # no more 'await'

my_system_prompt = "you are a smart virtual assistant who is perfect at summarizing a web page."
my_user_prompt = f"you will translate the content of this webpage {to_be_translated} into pidgin english"

my_messages = [
    {"role": "system", "content": my_system_prompt},
    {"role": "user", "content": my_user_prompt}
]

response = client.chat.completions.create(model="gpt-4o", messages=my_messages)
result = response.choices[0].message.content

display(Markdown(result))